In [30]:
# import opensim as osim
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from OA_utils.data_utils import *
import random
import pickle


Load data from file

In [56]:
# ── Configurable ─────────────────────────────────────────────────────────────
OA_data_dir  = "C:\\Users\\bakel\\Desktop\\GRFMuscleModel\\Old_Young_Walking_Data\\"
OA_filename  = 'Silder_OA_segs_normalized_filtered'
YA_data_dir  = "C:\\Users\\bakel\\Desktop\\GRFMuscleModel\\Old_Young_Walking_Data\\"
YA_filename  = 'Silder_YA_segs_normalized_filtered'
output_prefix = 'Silder_mixed'   # prefix for saved .npz files

SKIP_SUBJECTS = {'time_resampled', 'OA11'}

INPUT_KEYS  = ['grf_x', 'grf_y', 'grf_z', 'cop_x', 'cop_y', 'cop_z']
OUTPUT_KEYS = [
    'achilles', 'addbrev', 'addlong', 'addmagDist', 'addmagIsch', 'addmagMid',
    'addmagProx', 'bflh', 'bfsh', 'edl', 'ehl', 'fdl', 'fhl', 'gaslat',
    'gasmed', 'glmax1', 'glmax2', 'glmax3', 'glmed1', 'glmed2', 'glmed3',
    'glmin1', 'glmin2', 'glmin3', 'grac', 'iliacus', 'perbrev', 'perlong',
    'psoas', 'recfem', 'semimem', 'semiten', 'soleus', 'tibant', 'tibpost',
    'vasint', 'vaslat', 'vasmed',
]
ALL_KEYS = INPUT_KEYS + OUTPUT_KEYS
N_INPUTS = len(INPUT_KEYS)
# ─────────────────────────────────────────────────────────────────────────────

with open(OA_data_dir + OA_filename, 'rb') as f:
    OA_segs = pickle.load(f)
with open(YA_data_dir + YA_filename, 'rb') as f:
    YA_segs = pickle.load(f)

Visualize amount of data present for each subject

In [57]:
overall_OA_segs = 0
for subj, data in OA_segs.items():
    if subj in SKIP_SUBJECTS:
        continue
    num_segments = len(data["grf_y"])
    overall_OA_segs += num_segments
    print(f"{subj}: {num_segments} segments")
print(overall_OA_segs, 'OA segments overall')

OA1: 40 segments
OA2: 21 segments
OA4: 34 segments
OA5: 8 segments
OA7: 36 segments
OA8: 40 segments
OA9: 40 segments
OA10: 30 segments
OA12: 18 segments
OA13: 30 segments
OA14: 31 segments
OA17: 20 segments
OA18: 34 segments
OA19: 19 segments
OA20: 2 segments
OA22: 26 segments
OA23: 0 segments
OA24: 16 segments
OA25: 40 segments
485 OA segments overall


In [58]:
overall_YA_segs = 0
for subj, data in YA_segs.items():
    if subj in SKIP_SUBJECTS:
        continue
    num_segments = len(data["grf_y"])
    overall_YA_segs += num_segments
    print(f"{subj}: {num_segments} segments")
print(overall_YA_segs, 'YA segments overall')

Y1: 9 segments
Y2: 26 segments
Y4: 30 segments
Y5: 37 segments
Y6: 23 segments
Y7: 15 segments
Y8: 17 segments
Y9: 20 segments
Y10: 34 segments
Y11: 18 segments
Y12: 29 segments
Y13: 28 segments
Y14: 15 segments
Y15: 16 segments
Y16: 28 segments
Y17: 35 segments
Y18: 27 segments
Y19: 43 segments
Y20: 25 segments
Y21: 24 segments
Y22: 26 segments
525 YA segments overall


In [59]:
total_segs = overall_YA_segs + overall_OA_segs
print(total_segs, 'segments total')

1010 segments total


Split data by shuffling subjects

In [60]:
def count_total_segments(split_dict, key='grf_y'):
    return sum(len(split_dict[subj][key]) for subj in split_dict)

best_seed = None
best_err = float("inf")
best_counts = []
best_split = None
target_train, target_val, target_test = 0.8, 0.1, 0.1

for seed in range(40):
    OA_subjects = [s for s in OA_segs.keys() if s not in SKIP_SUBJECTS]
    random.seed(seed)
    OA_shuf = OA_subjects.copy()
    random.shuffle(OA_shuf)

    N = len(OA_shuf)
    train_end = int(N * 0.8)
    val_end   = int(N * 0.9)

    OA_train = {s: OA_segs[s] for s in OA_shuf[:train_end]}
    OA_val   = {s: OA_segs[s] for s in OA_shuf[train_end:val_end]}
    OA_test  = {s: OA_segs[s] for s in OA_shuf[val_end:]}

    YA_subjects = [s for s in YA_segs.keys() if s not in SKIP_SUBJECTS]
    random.seed(seed)
    YA_shuf = YA_subjects.copy()
    random.shuffle(YA_shuf)

    N = len(YA_shuf)
    train_end = int(N * 0.8)
    val_end   = int(N * 0.9)

    YA_train = {s: YA_segs[s] for s in YA_shuf[:train_end]}
    YA_val   = {s: YA_segs[s] for s in YA_shuf[train_end:val_end]}
    YA_test  = {s: YA_segs[s] for s in YA_shuf[val_end:]}

    train_segs = count_total_segments(OA_train) + count_total_segments(YA_train)
    val_segs   = count_total_segments(OA_val)   + count_total_segments(YA_val)
    test_segs  = count_total_segments(OA_test)  + count_total_segments(YA_test)

    train_p = train_segs / total_segs
    val_p   = val_segs   / total_segs
    test_p  = test_segs  / total_segs

    err = abs(train_p - target_train) + abs(val_p - target_val) + abs(test_p - target_test)

    if err < best_err:
        best_err = err
        best_seed = seed
        best_counts = (train_segs, val_segs, test_segs)
        best_split = {
            "OA_train_subjs": list(OA_train.keys()),
            "OA_val_subjs":   list(OA_val.keys()),
            "OA_test_subjs":  list(OA_test.keys()),
            "YA_train_subjs": list(YA_train.keys()),
            "YA_val_subjs":   list(YA_val.keys()),
            "YA_test_subjs":  list(YA_test.keys()),
        }

    print("Best seed:", best_seed)
    print("Counts (train/val/test):", best_counts)
    print("Ratios:",
          best_counts[0]/total_segs,
          best_counts[1]/total_segs,
          best_counts[2]/total_segs)
    print("Error:", best_err)

time_resampled = OA_segs.get("time_resampled")

Best seed: 0
Counts (train/val/test): (800, 100, 110)
Ratios: 0.7920792079207921 0.09900990099009901 0.10891089108910891
Error: 0.017821782178217824
Best seed: 0
Counts (train/val/test): (800, 100, 110)
Ratios: 0.7920792079207921 0.09900990099009901 0.10891089108910891
Error: 0.017821782178217824
Best seed: 0
Counts (train/val/test): (800, 100, 110)
Ratios: 0.7920792079207921 0.09900990099009901 0.10891089108910891
Error: 0.017821782178217824
Best seed: 0
Counts (train/val/test): (800, 100, 110)
Ratios: 0.7920792079207921 0.09900990099009901 0.10891089108910891
Error: 0.017821782178217824
Best seed: 0
Counts (train/val/test): (800, 100, 110)
Ratios: 0.7920792079207921 0.09900990099009901 0.10891089108910891
Error: 0.017821782178217824
Best seed: 0
Counts (train/val/test): (800, 100, 110)
Ratios: 0.7920792079207921 0.09900990099009901 0.10891089108910891
Error: 0.017821782178217824
Best seed: 0
Counts (train/val/test): (800, 100, 110)
Ratios: 0.7920792079207921 0.09900990099009901 0.108

In [61]:
def count_total_segments(split_dict, key='grf_y'):
    return sum(len(split_dict[subj][key]) for subj in split_dict)

OA_train_total = count_total_segments(OA_train)
YA_train_total = count_total_segments(YA_train)

OA_val_total = count_total_segments(OA_val)
YA_val_total = count_total_segments(YA_val)

OA_test_total = count_total_segments(OA_test)
YA_test_total = count_total_segments(YA_test)

print("Train (OA,YA):", OA_train_total, ',', YA_train_total)
print("Val (OA,YA):", OA_val_total,',', YA_val_total)
print("Test (OA,YA):", OA_test_total, ',',  YA_test_total)

Train (OA,YA): 353 , 424
Val (OA,YA): 74 , 35
Test (OA,YA): 58 , 66


In [49]:
def downsample_split(split_dict, target_n, seed=None, key='grf_y'):
    """
    Randomly downsample a subject dict so the total number of segments equals
    target_n.  Segments are dropped proportionally across subjects so no
    subject loses all its data.  Returns a new dict with the same structure.
    """
    rng = np.random.default_rng(seed)
    current_n = sum(len(data[key]) for data in split_dict.values())
    if target_n >= current_n:
        return split_dict

    subjects = list(split_dict.keys())
    counts = np.array([len(split_dict[s][key]) for s in subjects], dtype=float)
    allocated = np.maximum(1, np.round(counts / counts.sum() * target_n)).astype(int)

    # Fix any rounding drift so the total is exactly target_n
    diff = int(allocated.sum()) - target_n
    order = np.argsort(-allocated)
    for i in order:
        if diff == 0:
            break
        if diff > 0 and allocated[i] > 1:
            allocated[i] -= 1
            diff -= 1
        elif diff < 0:
            allocated[i] += 1
            diff += 1

    new_dict = {}
    for s, n_keep in zip(subjects, allocated):
        data = split_dict[s]
        n_total = len(data[key])
        idx = sorted(rng.choice(n_total, size=int(n_keep), replace=False))
        new_dict[s] = {signal: [vals[i] for i in idx] for signal, vals in data.items()}

    return new_dict


BALANCE_THRESHOLD = 0.2  # downsample if one dataset is >20% larger than the other

def balance_splits(a_dict, b_dict, split_name, seed=None):
    na = count_total_segments(a_dict)
    nb = count_total_segments(b_dict)
    ratio = abs(na - nb) / max(na, nb)
    if ratio <= BALANCE_THRESHOLD:
        print(f"{split_name}: balanced ({na} vs {nb}, {ratio:.1%} diff) — no downsampling needed.")
        return a_dict, b_dict

    target = min(na, nb)
    print(f"{split_name}: imbalanced ({na} vs {nb}, {ratio:.1%} diff) → downsampling larger split to {target}")
    if na > nb:
        return downsample_split(a_dict, target, seed=seed), b_dict
    else:
        return a_dict, downsample_split(b_dict, target, seed=seed)


OA_train, YA_train = balance_splits(OA_train, YA_train, "Train", seed=best_seed)
OA_val,   YA_val   = balance_splits(OA_val,   YA_val,   "Val",   seed=best_seed)
OA_test,  YA_test  = balance_splits(OA_test,  YA_test,  "Test",  seed=best_seed)

print("\nPost-balance counts:")
print(f"  Train  OA={count_total_segments(OA_train)}  YA={count_total_segments(YA_train)}")
print(f"  Val    OA={count_total_segments(OA_val)}   YA={count_total_segments(YA_val)}")
print(f"  Test   OA={count_total_segments(OA_test)}   YA={count_total_segments(YA_test)}")

Train: imbalanced (353 vs 8330, 95.8% diff) → downsampling larger split to 353
Val: imbalanced (74 vs 1249, 94.1% diff) → downsampling larger split to 74
Test: imbalanced (58 vs 1005, 94.2% diff) → downsampling larger split to 58

Post-balance counts:
  Train  OA=353  YA=353
  Val    OA=74   YA=74
  Test   OA=58   YA=58


In [62]:
def dict_to_array(split_dict):
    packed_segments = []
    for subj, data in split_dict.items():
        num_segs = len(data[INPUT_KEYS[0]])
        for i in range(num_segs):
            sample = np.column_stack([data[k][i] for k in ALL_KEYS])
            packed_segments.append(sample)
    return np.array(packed_segments)

In [63]:
def get_subject_ranges(split_dict):
    ranges = {}
    start_idx = 0
    for subj, data in split_dict.items():
        n = len(data[INPUT_KEYS[0]])
        end_idx = start_idx + n
        ranges[subj] = (start_idx, end_idx)
        start_idx = end_idx
    return ranges

In [64]:
OA_test_ranges = get_subject_ranges(OA_test)
print(OA_test_ranges)

{'OA12': (0, 18), 'OA9': (18, 58)}


In [65]:
OA_train_arr = dict_to_array(OA_train)
OA_val_arr   = dict_to_array(OA_val)
OA_test_arr  = dict_to_array(OA_test)

YA_train_arr = dict_to_array(YA_train)
YA_val_arr   = dict_to_array(YA_val)
YA_test_arr  = dict_to_array(YA_test)

train_arr = np.concatenate([OA_train_arr, YA_train_arr], axis=0)
val_arr   = np.concatenate([OA_val_arr,   YA_val_arr],   axis=0)
test_arr  = np.concatenate([OA_test_arr,  YA_test_arr],  axis=0)

print("Train:", train_arr.shape)
print("Val:",   val_arr.shape)
print("Test:",  test_arr.shape)

Train: (777, 100, 44)
Val: (109, 100, 44)
Test: (124, 100, 44)


In [66]:
X_train, y_train = train_arr[:, :, :N_INPUTS], train_arr[:, :, N_INPUTS:]
X_val,   y_val   = val_arr[:,   :, :N_INPUTS], val_arr[:,   :, N_INPUTS:]
X_test,  y_test  = test_arr[:,  :, :N_INPUTS], test_arr[:,  :, N_INPUTS:]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

X_train shape: (777, 100, 6)
y_train shape: (777, 100, 38)
X_val shape:   (109, 100, 6)
y_val shape:   (109, 100, 38)
X_test shape:  (124, 100, 6)
y_test shape:  (124, 100, 38)


In [67]:
np.savez(OA_data_dir + f'{output_prefix}_train_data.npz', X_train=X_train, y_train=y_train)
np.savez(OA_data_dir + f'{output_prefix}_val_data.npz',   X_val=X_val,     y_val=y_val)
np.savez(OA_data_dir + f'{output_prefix}_test_data.npz',  X_test=X_test,   y_test=y_test)